# SCD Type 3 — Previous Value Column

**SCD Type 3** keeps a **limited** history by adding extra columns to the same row:  
a `previous_name` column alongside the `current_name`, and the date of the last change.

Only **one step back** is preserved. If a value changes twice, the original is lost.

### When to use Type 3
- You need to compare "current vs previous" without a full history.
- Storage is very constrained and you can accept limited history.
- The dimension changes rarely and only one prior value matters to the business.

### Table structure
| Column | Purpose |
|---|---|
| `id` | Natural/business key |
| `current_name` | The latest value |
| `previous_name` | The value before the last change (`NULL` if never changed) |
| `effective_date` | When `current_name` became active |
| `previous_date` | When `previous_name` was active (= original load date) |

### Techniques covered
1. Create the Type 3 Delta table
2. Initial load from source CSV
3. MERGE — shift current → previous, write new current
4. Query current vs previous values side by side
5. Apply a second change to illustrate the one-step-back limitation
6. Delta history

## Setup

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

DATA_ROOT = "/Volumes/population_metrics/landing/datasets"

from pyspark.sql import functions as F, Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from delta.tables import DeltaTable

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

## 1. Read the source CSV

In [ ]:
CSV_PATH = f"{DATA_ROOT}/countries_dataset/countries_dataset/csv_data/country_regions/country_regions.csv"

source_schema = StructType([
    StructField("id",   IntegerType(), False),
    StructField("name", StringType(),  True)
])

source_df = spark.read.format("csv").schema(source_schema).option("header", True).load(CSV_PATH)
source_df.show()

## 2. Create the SCD Type 3 Delta table

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS population_metrics.default.scd3_country_regions (
        id              INT     NOT NULL,
        current_name    STRING,
        previous_name   STRING,
        effective_date  TIMESTAMP,
        previous_date   TIMESTAMP
    )
    USING DELTA
    COMMENT 'SCD Type 3 — keeps current and one previous value per row'
""")
print("Table ready.")

## 3. Initial load

All rows start with `previous_name = NULL` and `previous_date = NULL` — no prior value exists yet.

In [ ]:
row_count = spark.table("population_metrics.default.scd3_country_regions").count()

if row_count == 0:
    initial_df = source_df.select(
        F.col("id"),
        F.col("name").alias("current_name"),
        F.lit(None).cast("string").alias("previous_name"),
        F.current_timestamp().alias("effective_date"),
        F.lit(None).cast("timestamp").alias("previous_date")
    )
    initial_df.write.format("delta").mode("append").saveAsTable("population_metrics.default.scd3_country_regions")
    print(f"Seeded {initial_df.count()} rows.")
else:
    print(f"Table already has {row_count} rows — skipping initial load.")

spark.table("population_metrics.default.scd3_country_regions").orderBy("id").show(truncate=False)

## 4. MERGE — the Type 3 update pattern

The key difference from Type 1: before overwriting `current_name` we **shift it into `previous_name`**.

For new ids (`WHEN NOT MATCHED`) we insert normally with `previous_name = NULL`.

Simulated change: America → *North & South America*, Antarctica (id=60) added.

In [ ]:
updated_rows = [
    Row(id=10, name="North & South America"),  # changed
    Row(id=20, name="Europe"),
    Row(id=30, name="Asia"),
    Row(id=40, name="Oceania"),
    Row(id=50, name="Africa"),
    Row(id=60, name="Antarctica"),              # new
]
updated_df = spark.createDataFrame(updated_rows, schema=source_schema)

scd3 = DeltaTable.forName(spark, "population_metrics.default.scd3_country_regions")

scd3.alias("target").merge(
    updated_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(
    condition="target.current_name <> source.name",   # only act when something changed
    set={
        # Shift current → previous before overwriting
        "previous_name":  "target.current_name",
        "previous_date":  "target.effective_date",
        # Write new current
        "current_name":   "source.name",
        "effective_date": "current_timestamp()"
    }
).whenNotMatchedInsert(
    values={
        "id":             "source.id",
        "current_name":   "source.name",
        "previous_name":  "NULL",
        "effective_date": "current_timestamp()",
        "previous_date":  "NULL"
    }
).execute()

print("Merge complete.")
spark.table("population_metrics.default.scd3_country_regions").orderBy("id").show(truncate=False)

## 5. Query current vs previous side by side

This is the main analytical benefit of Type 3 — a simple SELECT gives you the current and prior name in one row, with no joins or window functions needed.

In [ ]:
spark.sql("""
    SELECT
        id,
        current_name,
        previous_name,
        CASE WHEN previous_name IS NOT NULL THEN 'Changed' ELSE 'Unchanged' END AS status,
        effective_date  AS current_since,
        previous_date   AS was_current_since
    FROM population_metrics.default.scd3_country_regions
    ORDER BY id
""").show(truncate=False)

## 6. The one-step-back limitation

Apply a **second rename** to `id = 10` to demonstrate that the original value "America" is lost forever once `previous_name` gets overwritten.

In [ ]:
second_change = spark.createDataFrame(
    [Row(id=10, name="The Americas")],   # second rename
    schema=source_schema
)

scd3.alias("target").merge(
    second_change.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(
    condition="target.current_name <> source.name",
    set={
        "previous_name":  "target.current_name",
        "previous_date":  "target.effective_date",
        "current_name":   "source.name",
        "effective_date": "current_timestamp()"
    }
).execute()

print("Second change applied to id=10.")
spark.sql("""
    SELECT id, current_name, previous_name, effective_date
    FROM population_metrics.default.scd3_country_regions
    WHERE id = 10
""").show(truncate=False)

print("⚠️  'America' (the original value) is now permanently lost — only one step back is kept.")

## 7. Comparison: Type 1 vs Type 2 vs Type 3

| | Type 1 | Type 2 | Type 3 |
|---|---|---|---|
| History kept | None | Full (unlimited) | One step |
| Row count per key | 1 | Grows with changes | Always 1 |
| Query complexity | Trivial | Requires filter/join | Trivial |
| Storage cost | Low | High | Low |
| Use case | Corrections | Full audit trail | Before/after reports |

## 8. Delta transaction log

In [ ]:
spark.sql("""
    DESCRIBE HISTORY population_metrics.default.scd3_country_regions
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## 9. (Optional) Cleanup

In [ ]:
# Uncomment to reset
# spark.sql("DROP TABLE IF EXISTS population_metrics.default.scd3_country_regions")
# print("Table dropped.")